# testing swap pricing and sub functions setep by step

In [44]:
%load_ext autoreload
%autoreload 2
#imports
from rivapy.pricing.interest_rate_swap_pricing import InterestRateSwapPricer
from rivapy.instruments.ir_swap_specification import InterestRateSwapSpecification, IrFixedLegSpecification, IrFloatLegSpecification
from rivapy.instruments.notional_structure import NotionalStructure, ConstNotionalStructure, VariableNotionalStructure, ResettingNotionalStructure
from rivapy.pricing.pricing_data import (
    InterestRateSwapFloatLegPricingData_rivapy,
    InterestRateSwapLegPricingData_rivapy,
    InterestRateSwapPricingData_rivapy,
)
from rivapy.pricing.pricing_request import InterestRateSwapPricingRequest
from rivapy.pricing.interest_rate_swap_pricing import InterestRateSwapPricer

import datetime as dt
from rivapy.marketdata.curves import DiscountCurve
from rivapy.tools.enums import InterpolationType, ExtrapolationType
import math


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
# Discount curve - we use these discount factors to get the present values of both the fixed and floating leg as well as

object_id = "TEST_DC"
refdatedc = dt.datetime(2017, 1, 1)
days_to_maturity = [180, 360, 540]
dates = [refdatedc + dt.timedelta(days=d) for d in days_to_maturity]
# discount factors from constant rate
rates = [0.10, 0.105, 0.11]
df = [math.exp(-r * d / 360) for r, d in zip(rates, days_to_maturity)]
dc = DiscountCurve(
    id=object_id, refdate=refdatedc, dates=dates, df=df, interpolation=InterpolationType.LINEAR, extrapolation=ExtrapolationType.LINEAR
)

In [46]:
#test market data
ref_date = dt.datetime(2017, 1, 1)
dates = [dt.datetime(2017, 1, 1), dt.datetime(2018, 7, 1)]
df = [1.0, 0.9900633419771339]
dc = DiscountCurve(id=object_id, refdate=refdatedc, dates=dates, df=df, interpolation=InterpolationType.LINEAR, extrapolation=ExtrapolationType.LINEAR)

In [47]:
# Create the vectors defining the statdates, enddates, paydates and reset dates
days_to_maturity = [0, 180, 360, 540]
dates = [dt.datetime(2017, 1, 1) + dt.timedelta(d) for d in days_to_maturity]

startdates = dates[:-1]
enddates = dates[1:]
paydates = enddates
resetdates = startdates
refdate = dates[0]

In [48]:
# fixedleg = IrFixedLegSpecification(0.08, notionals, startdates, enddates, paydates, 'EUR', 'Act360')
fixed_leg = IrFixedLegSpecification(
    fixed_rate=0.08,
    obj_id="dummy_fixed_leg",
    notional=100.0,
    start_dates=startdates,
    end_dates=enddates,
    pay_dates=paydates,
    currency="EUR",
    day_count_convention="Act360",
)

spread = 0.00

ns = ConstNotionalStructure(100.0)
# floatleg = IrFloatLegSpecification(notionals, resetdates, startdates, enddates, paydates, 'EUR', 'test_udl',
#                                   'Act360', spread)

float_leg = IrFloatLegSpecification(
    obj_id="dummy_float_leg",
    notional=ns,
    reset_dates=resetdates,
    start_dates=startdates,
    end_dates=enddates,
    rate_start_dates=startdates,
    rate_end_dates=enddates,
    pay_dates=paydates,
    currency="EUR",
    udl_id="test_udl_id",
    fixing_id="test_fixing_id",
    day_count_convention="Act360",
    spread=spread,
)

maturity_date = refdate + dt.timedelta(600)
# ir_swap = InterestRateSwapSpecification('TEST_SWAP', 'DBK', 'COLLATERALIZED', 'EUR', paydates[-1], fixedleg, floatleg)
ir_swap = InterestRateSwapSpecification(
    obj_id="dummy_swap",
    notional=ns,
    issue_date=refdate,
    maturity_date=maturity_date,
    pay_leg=fixed_leg,
    receive_leg=float_leg,
    currency="EUR",
    day_count_convention="EUR",
    issuer="DBK",
    securitization_level="COLLATERALIZED",
)

In [49]:
#Pricing the fixed leg

fixed_PV = InterestRateSwapPricer.price_leg(
        refdate, dc, dc, None, fixed_leg, None, 0  # discount  # forward/fixing  # fx_fwd  # leg_spec  # fixing table  # fixing grace period
    )



generating cashflow table for FIXED leg
----------------------------------------------------------
DEBUG: price leg pv values
3.932281691206147
3.9193579029602397
3.9064341147143318


In [50]:
#pricing float leg
float_PV = InterestRateSwapPricer.price_leg(
        refdate, dc, dc, None, float_leg, None, 0  # discount  # forward/fixing  # fx_fwd  # leg_spec  # fixing table  # fixing grace peropd
    )



generating cashflow table for FLOAT leg
notional 0:100.0 for 2017-06-30 00:00:00
swap calculating fwd_rate for floating leg
7777777777777777777777777777777777777777777777
Debugging rivapy_valueFWD: x (yearfrac), then y (df) lists
[0.0, 1.4958904109589042]
[1.0, 0.9900633419771339]
7777777777777777777777777777777777777777777777
Debugging rivapy_valueFWD: x (yearfrac), then y (df) lists
[0.0, 1.4958904109589042]
[1.0, 0.9900633419771339]
notional 1:100.0 for 2017-12-27 00:00:00
swap calculating fwd_rate for floating leg
7777777777777777777777777777777777777777777777
Debugging rivapy_valueFWD: x (yearfrac), then y (df) lists
[0.0, 1.4958904109589042]
[1.0, 0.9900633419771339]
7777777777777777777777777777777777777777777777
Debugging rivapy_valueFWD: x (yearfrac), then y (df) lists
[0.0, 1.4958904109589042]
[1.0, 0.9900633419771339]
notional 2:100.0 for 2018-06-25 00:00:00
swap calculating fwd_rate for floating leg
7777777777777777777777777777777777777777777777
Debugging rivapy_valueFWD: x 

In [51]:
print(f"float pv: {float_PV}")
print(f"fixed_pv: {fixed_PV}")
print(f"price = {float_PV - fixed_PV}")

float pv: 0.9801955758587282
fixed_pv: 11.758073708880719
price = -10.777878133021991


In [52]:
#Computeing fixed leg annuity, i.e. if fixed ratet = 1
pricing_params = {"set_rate": True, "desired_rate": 1.0}
fixed_leg_annuity = InterestRateSwapPricer.price_leg(
        refdate, dc, dc, None, fixed_leg, None, 0, pricing_params  # discount  # forward/fixing  # fx_fwd  # leg_spec  # fixing table  # fixing grace period
    )

print(f"annuity: {fixed_leg_annuity}")

generating cashflow table for FIXED leg
----------------------------------------------------------
DEBUG: price leg pv values
49.15352114007684
48.991973787003
48.83042643392915
annuity: 146.975921361009


In [53]:
#compute fair rate swap i.e. float_pv = fixed_pv
# where we want the fixed rate that makes that equation true

fair_swap_rate = float_PV / fixed_leg_annuity
print(f"fair swap rate: {fair_swap_rate}")

fair swap rate: 0.006669089513316449
